In [ ]:
from pathlib import Path
import json
import numpy as np
from skimage.io import imsave
from nd2 import ND2File
import warnings
import pandas as pd
from sklearn.cluster import DBSCAN
from skimage.draw import ellipse_perimeter
from skimage.exposure import rescale_intensity
from skimage.color import gray2rgb
from skimage.restoration import rolling_ball
from matplotlib import colors as mcolors
from scipy.optimize import OptimizeWarning

from calmutils.localization import refine_point_lsq, detect_dog
from calmutils.localization.util import sigma_to_full_width_at_quantile, full_width_at_quantile_to_sigma
from calmutils.imageio.nd2_helpers import get_pixel_size

def load_channels_from_nd2(file_path, channels, position=None):

    res = {}

    # do a few sanity checks concerning multi-position files
    num_positions = get_num_positions_nd2(file_path)
    if num_positions > 1 and position is None:
        raise ValueError(f'Multiple xy positions in file {file_path}, please specify which one to load via the position parameter.')
    if num_positions == 1 and position not in [0, None]:
        warnings.warn(f'File {file_path} does not contain multiple xy positions, will load the only available position instead of specified position {position}.')

    with ND2File(file_path) as reader:

        # nice OC name without whitespace
        channel_names = list(map(lambda s: s.channel.name.strip().replace(' ', '-'), reader.metadata.channels))
            
        for channel in channels:

            # try to find specified channel, otherwise warn and list available channels
            try:
                channel_idx = channel_names.index(channel)
            except ValueError:
                warnings.warn(f'channel {channel} not found in file {file_path}. available channels: {channel_names}')
                continue
            
            if num_positions == 1:
                img = np.array(reader.to_dask()[:, channel_idx])
            else:
                img = np.array(reader.to_dask()[position, :, channel_idx])
            res[channel] = img
        # invert xyz voxel size to zyx to match img array
        pixel_size = reader.voxel_size()[::-1]
    
    return res, pixel_size

def get_num_positions_nd2(in_file):
    """
    get the number of xy-positions / tiles in an nd2 file, will return 1 if file is just a single (potentially multichannel) stack
    """
    with ND2File(in_file) as reader:
        return reader.sizes['P'] if 'P' in reader.sizes else 1

def imsave_nowarnings(file, img, **kwargs):
    # catch low contrast warning
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        imsave(file, img, **kwargs)

def filter_clustering(blobs, pixel_size, expected_size, cluster_reject_distance, cluster_reject_n_spots):
    # drop sigma columns produced by blob_log/blob_dog
    blobs_just_coords = blobs * pixel_size / expected_size
    # spots that receive class -1 in DBSCAN := not in cluster
    # NOTE: explicitly setting algorithm='kd_tree' was necessary to avoid issues in multithreaded processing of files
    single_spot_idx = DBSCAN(cluster_reject_distance, min_samples=cluster_reject_n_spots, algorithm='kd_tree').fit_predict(blobs_just_coords) == -1
    return blobs[single_spot_idx], single_spot_idx


def refine_points(image, points, sigma_expected=None, max_log2_deviation_from_expected_size=None):
    """
    Parameters
    ----------
    image: ndarray image to refine points in
    points: (n, image.ndim) array of n candidate points to refine
    """

    points_refined = []
    sigmas_refined = []
    minmax_refined = []

    # if expected size is given, cut until expected full-with at 5% intenisty of gaussian
    cutregion = np.round(sigma_to_full_width_at_quantile(sigma_expected, 0.05) / 2).astype(int) if sigma_expected is not None else None

    for blob in points:

        # do Gaussian fit, ignore warnings about failed optimization -> we will skip those blobs
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', (OptimizeWarning, RuntimeWarning))
            pos_refined, fit = refine_point_lsq(image, blob, cutregion)
        
        # skip if fit not possible or negative sigma
        if fit is None:
            continue
        fit, _ = fit
        if np.any(fit[-3:] < 0) or np.any(np.isnan(fit)):
            continue

        sigma = fit[-3:]

        # discard spot if the sigma deviates too much from expected size
        # we calculate the mean absolute log2 of ratio expected/fit and discard if bigger than our threshold
        if sigma_expected is not None and max_log2_deviation_from_expected_size is not None:
            if np.abs(np.log2(np.array(sigma_expected) / np.array(sigma))).mean() > max_log2_deviation_from_expected_size:
                continue

        points_refined.append(pos_refined)
        sigmas_refined.append(sigma)
        minmax_refined.append(fit[:2])

    points_refined = np.array(points_refined)
    return points_refined, np.array(sigmas_refined), np.array(minmax_refined)

def get_spot_visualization_projection(img, blobs, sigmas):

    # color to plot ellipse in
    ellipse_color = np.array(mcolors.hex2color(mcolors.XKCD_COLORS['xkcd:sea green']))
    # how much to expand the ellipse with radius = full width at tenth maximum
    ellipse_expansion_factor = 4
    # ellipse line width
    ellipse_line_width = 3

    img_projected = img.max(axis=0)
    proj_rgb = gray2rgb(rescale_intensity(img_projected, in_range=tuple(np.quantile(img_projected, (0.02, 0.9999))), out_range='float32'))

    for blob, sigma in zip(blobs, sigmas):
        
        # get position and sigma of blob in yx
        yx = blob[1:].astype(int)
        sy_sx = sigma[1:]

        # to radius of ellipse (based on full width at tenth maximum times expansion factor)
        ry_rx = (sigma_to_full_width_at_quantile(sy_sx, 0.1) / 2 * ellipse_expansion_factor).astype(int)
        
        # line width: draw single pixel ellipse at radius + 0, +1, ...
        for i in range(ellipse_line_width):
            proj_rgb[tuple(ellipse_perimeter(*yx, *(ry_rx+i), shape=proj_rgb.shape))] = ellipse_color

    proj_rgb = (proj_rgb * 255).astype(np.uint8)

    return proj_rgb



def blobs_to_df(blobs_i, file_path, position, sigmas: dict, minmax):

    pixel_size = get_pixel_size(file_path)

    df = pd.DataFrame()
    for channel_name, blobs_ii in blobs_i.items():

        # NOTE: reshape to prevent errors on empty results ((0,3)-array)
        # should also result in column names not being dropped
        blobs_ii = blobs_ii.reshape((-1, 3))

        df_i = pd.DataFrame({
            'spot_idx': np.arange(len(blobs_ii), dtype=int),
            'image_file': file_path,
            'position_idx': position,
            'channel': channel_name,
            **dict(zip('zyx', blobs_ii.T)), 
            **dict(zip(['z_micron', 'y_micron', 'x_micron'], (blobs_ii * pixel_size).T))
            })
        
        sigmas_i = sigmas.get(channel_name, None)
        minmax_i = minmax.get(channel_name, None)

        # reshape to keep track of dimensionality even for empty results, see above
        if sigmas_i is not None:
            sigmas_i = sigmas_i.reshape((-1, 3))
        if minmax_i is not None:
            minmax_i = minmax_i.reshape((-1, 2))
        
        if sigmas_i is not None:
            for col_name, vals in zip(['sigma_z', 'sigma_y', 'simga_x'], sigmas_i.T):
                df_i[col_name] = vals
        if sigmas_i is not None:
            for col_name, vals in zip(['sigma_z_micron', 'sigma_y_micron', 'simga_x_micron'], (sigmas_i * pixel_size).T):
                df_i[col_name] = vals
        if minmax_i is not None:
            for col_name, vals in zip(['gauss_fit_min', 'gauss_fit_height'], minmax_i.T):
                df_i[col_name] = vals

        df = pd.concat([df, df_i], ignore_index=True)
    
    return df

In [ ]:
# path containing files to visualize
in_path = '/Volumes/agl_data/NanoFISH/Gabi/GS666_tetraspeck_on_cells_1-50'

# subdirectory containing input data
# leave empty ('') if image files are directly in in_path
in_subdirectory = 'raw'

# default: put results in subdirectory called 'spot-detection'
out_subdirectory = 'spot-detection'

# which channels to include
channels_to_include = ['561-CSU-W1', '640-CSU-W1']

# DoG threshold for all channels
thresholds_dog = 0.02
# Alternative: threshold_log can be a dictionary containing a separate threshold for each channel
thresholds_dog = {
    '561-CSU-W1': 12,
    '640-CSU-W1': 26
}
# consider DoG threshold relative to maximum filter response?
# absolute thresholds seem more robust?
threshold_dog_relative = False

# Thresholds in intensity
# NOTE: will be measured in slightly blurred (and background-subtracted) images, so consider that when setting threshold
# NOTE: did not seem to improve results, therefore set to None again
thresholds_intensity = None
# thresholds_intensity = {
#     '561-CSU-W1': 100.0,
#     '640-CSU-W1': 170.0
# }

# whether to do background subtraction via Rolling Ball algorithm and ball radius
# NOTE: helps when thresholding raw intensity, but not much when thresholding DoG response
do_background_subtraction = False
background_subtraction_radius = 5

# refine points via Gaussian fit?
do_gaussian_fit = True

# expected size (zyx, in microns)
expected_size = [0.7, 0.35, 0.35]

# how many peaks to detect at max
# NOTE: this is mainly a fail-safe to prevent needlessly long computations when thresholds are set incorrectly
max_num_peaks = 2000

# maximum log2 fold deviation from expected size
# can be used to filter out large objects, e.g. 1: fitted gaussian has to be within 0.5 - 2x expected size
# set to None to skip
max_log2_deviation_from_expected_size = 1

# cluster rejection via DBSCAN clustering
# spots that lie in clusters in which at least cluster_reject_n_spots spots lie within cluster_reject_distance are ignored
# cluster_reject_distance is in units of expected size (i.e. 5.0: are considered to cluster if their distance is < 5*expected_size)
cluster_reject_distance = 5.0
cluster_reject_n_spots = 5

save_visualization = True

# how many images to process in parallel
num_threads = 16

In [ ]:
# make Paths
in_path = Path(in_path)
out_path = in_path / out_subdirectory

# make dict with same threshold for all channels
if not isinstance(thresholds_dog, dict):
    thresholds_dog = {channel: thresholds_dog for channel in channels_to_include}
if not isinstance(thresholds_intensity, dict):
    thresholds_intensity = {channel: thresholds_intensity for channel in channels_to_include}

expected_size = np.array(expected_size)

parameter_log = {
    'in_path': str(in_path),
    'in_subdirectory': in_subdirectory,
    'channels_to_include': channels_to_include,
    'thresholds_dog': thresholds_dog,
    'thresholds_intensity': thresholds_intensity,
    'do_gaussian_fit': do_gaussian_fit,
    'expected_size': list(expected_size),
    'cluster_reject_distance': cluster_reject_distance,
    'cluster_reject_n_spots': cluster_reject_n_spots,
    'max_log2_deviation_from_expected_size': max_log2_deviation_from_expected_size,
    'max_num_peaks': max_num_peaks,
    'do_background_subtraction': do_background_subtraction,
    'background_subtraction_radius': background_subtraction_radius,
    'threshold_dog_relative': threshold_dog_relative
}

# get all nd2 files in in_path
in_files = sorted((Path(in_path) / in_subdirectory).glob('*.nd2'))

# TODO: check if anything is multistack and print?

# show for verification
in_files

In [ ]:
from concurrent.futures import ThreadPoolExecutor

if not out_path.exists():
    out_path.mkdir(parents=True)

visualization_path = out_path / 'quick_result_visualization'
if save_visualization and not visualization_path.exists():
    visualization_path.mkdir(parents=True)


def load_and_detect_spots(in_file, position_idx, channels_to_include,
                          expected_size, thresholds_dog, do_visualization, cluster_reject_distance, cluster_reject_n_spots,
                          do_gaussian_fit, max_log2_deviation_from_expected_size, thresholds_intensity, max_num_peaks,
                          do_background_subtraction, background_subtraction_radius, threshold_dog_relative,
                          **kwargs):

    images, pixel_size = load_channels_from_nd2(in_file, channels_to_include, position_idx)

    # print(f'loaded {position_idx}')

    # sigma for expected size in pixels
    sigma_expected = full_width_at_quantile_to_sigma(expected_size) / pixel_size

    # results will be dicts mapping channel names to result values
    blobs = {}
    sigmas = {}
    minmax = {}
    projections = {}

    for channel_name, img in images.items():

        # planewise background subtraction via Rolling Ball
        if do_background_subtraction:
            background_estimate = np.stack([rolling_ball(plane, radius=background_subtraction_radius, num_threads=1) for plane in img])
            img = img - background_estimate

        # custom DoG with maximum number of peaks, thresholding in raw intensity
        blobs_i = detect_dog(img, 
                threshold=thresholds_dog[channel_name], threshold_intensity=thresholds_intensity[channel_name],
                sigma=sigma_expected, max_num_peaks=max_num_peaks, threshold_relative=threshold_dog_relative)

        # we have only one sigma, repeat for each detection
        sigmas_i = np.tile(sigma_expected, len(blobs_i)).reshape((-1, img.ndim))

        if do_gaussian_fit:
            blobs_i, sigmas_i, minmax_i = refine_points(img, blobs_i, sigma_expected, max_log2_deviation_from_expected_size)

        if len(blobs_i) > cluster_reject_n_spots:
            blobs_i, single_spot_idx = filter_clustering(blobs_i, pixel_size, expected_size, cluster_reject_distance, cluster_reject_n_spots)
            sigmas_i = sigmas_i[single_spot_idx]
            if do_gaussian_fit:
                minmax_i = minmax_i[single_spot_idx]
        
        blobs[channel_name] = blobs_i

        if do_gaussian_fit:
            sigmas[channel_name] = sigmas_i
            minmax[channel_name] = minmax_i

        # generate projection with detections visualized
        if do_visualization:
            visualization_projection = get_spot_visualization_projection(img, blobs_i, sigmas_i)
            projections[channel_name] = visualization_projection
    
    return blobs, projections, sigmas, minmax


# call main function load_and_detect_spots multithreaded
with ThreadPoolExecutor(num_threads) as tpe:

    futures = []
    for in_file in in_files:
        num_positions = get_num_positions_nd2(in_file)
        for position_idx in range(num_positions):
            # use common parameters as well as image and position idx as kwargs to load_and_detect_spots
            # do_visualization is added here as well, as it is not saved to parameter_log
            kwargs = parameter_log | {'in_file': in_file, 'position_idx': position_idx, 'do_visualization': save_visualization}
            futures.append(tpe.submit(load_and_detect_spots, **kwargs))
        
    blobs = []
    visualization_projections = []
    sigmas = []
    minmaxs = []

    future_iter = iter(futures)
    for in_file in in_files:
        num_positions = get_num_positions_nd2(in_file)
        for position_idx in range(num_positions):
            f = next(future_iter)
            blobs_i, visualization_projections_i, sigmas_i, minmax_i = f.result()
            for channel_name, b in blobs_i.items():
                print(f'{in_file}{"" if num_positions == 1 else f"(stack {position_idx})"}: number of detected blobs in {channel_name}: {len(b)}')
            blobs.append(blobs_i)
            visualization_projections.append(visualization_projections_i)
            sigmas.append(sigmas_i)
            minmaxs.append(minmax_i)

with open(out_path / 'spot_detection_parameters.json', 'w') as fd:
    json.dump(parameter_log, fd, indent=1) 

i = 0
for in_file in in_files:
    num_positions = get_num_positions_nd2(in_file)
    for position_idx in range(num_positions):

        df = blobs_to_df(blobs[i], in_file, position_idx, sigmas[i], minmaxs[i])
        out_file = out_path / (in_file.stem + ('' if num_positions == 1 else f'_stack{position_idx}' ) + '_spot-detection.csv')
        df.to_csv(out_file, index=None)

        if save_visualization:
            for channel_name, visualization_projection in visualization_projections[i].items():
                # make filepath for output
                outfile_visualization = visualization_path / (in_file.stem + ('' if num_positions == 1 else f'_stack{position_idx}' ) + f'_{channel_name}_spot-detection.png')            
                imsave_nowarnings(str(outfile_visualization), visualization_projection)
        i += 1